In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

import json
import math
from pathlib import Path

import numpy as np
from scipy.stats import kendalltau

# ------------------------------------------------------------------ #
# Same ground-truth parsing as eval_metrics.py: None/NaN handling and the
# no-N/H-for-proline and no-CB-for-glycine exclusions, identical to training/validation.
# (same functions as rmse_comparison_results_5seedens.ipynb)
# ------------------------------------------------------------------ #
def parse_shift_list(lst, sequence, atom):
    vals, mask = [], []
    for i, x in enumerate(lst):
        residue = sequence[i]
        bad = False
        if x is None:
            bad = True
        elif isinstance(x, str):
            xs = x.strip().lower()
            if xs in {"none", "null", "nan", "inf", "+inf", "-inf"}:
                bad = True
            else:
                try:
                    x = float(x)
                except Exception:
                    bad = True
        if not bad and isinstance(x, (int, float)):
            if not math.isfinite(float(x)):
                bad = True
        if atom in ("N", "H") and residue == "P":
            bad = True
        if atom == "CB" and residue == "G":
            bad = True
        if bad:
            vals.append(0.0)
            mask.append(False)
        else:
            vals.append(float(x))
            mask.append(True)
    return vals, mask


def load_ground_truth(gt_path: Path, atom: str):
    """entry_ID (without the bmr prefix) -> (shift list, mask list)."""
    with open(gt_path) as f:
        rbmrb = json.load(f)
    out = {}
    for ref_id, shift_lst in rbmrb[atom].items():
        if all(x is None or (isinstance(x, str) and x.lower() in ("none", "null"))
               for x in shift_lst):
            continue
        sequence = rbmrb["sequence"][ref_id]
        shift, mask = parse_shift_list(shift_lst, sequence, atom=atom)
        entry = rbmrb["entry_ID"][ref_id].lower().removeprefix("bmr")
        out[entry] = (shift, mask)
    return out


def load_pred(path):
    """In predictions_N_seed_ensemble_*.json, "pred" is the per-seed [n_seeds, n_res] matrix;
    evaluation uses the precomputed per-residue seed mean "mean" (with "mask")."""
    with open(path) as f:
        d = json.load(f)
    return d["mean"], d["mask"]


def build_per_protein(pred_path, gt_path, atom):
    """entry -> (pred_vals, gt_vals), aligned exactly as in eval_metrics.py:
    only the prediction's own mask intersected with the GT None/Pro-N-H/Gly-CB mask.
    The tag/flag/extra masks shipped with each test set (N_mask, *_flag, ...) were already folded
    into the prediction mask when predictions were generated, so they are not applied again."""
    pred_dict, mask_dict = load_pred(pred_path)
    gt = load_ground_truth(Path(gt_path), atom=atom)

    per_protein = {}
    skipped = []
    for entry, (true_shift, true_mask) in gt.items():
        if entry not in pred_dict:
            skipped.append(entry)
            continue
        pred_shift = pred_dict[entry]
        pred_mask = mask_dict[entry]
        n = min(len(true_shift), len(true_mask), len(pred_shift), len(pred_mask))
        combined_mask = np.array(true_mask[:n], dtype=bool) & np.array(pred_mask[:n], dtype=bool)
        if not combined_mask.any():
            skipped.append(entry)
            continue
        per_protein[entry] = {
            "pred": np.asarray(pred_shift[:n], dtype=float)[combined_mask],
            "gt":   np.asarray(true_shift[:n], dtype=float)[combined_mask],
        }
    return per_protein, skipped

In [ ]:
RESULTS_DIR = Path("../../data/analysis/Fig2/results")
ATOM = "N"
MIN_VALID_RES = 3  # Kendall's tau is meaningless for very few residues; skip those proteins

models = ["lacs", "panav", "noref", "noref_whole"]
model_pred_path = {m: RESULTS_DIR / f"predictions_N_seed_ensemble_10seed_{m}.json" for m in models}
gt_path = RESULTS_DIR / "test_panav.json"   # all three models are compared on the PANAV test set

per_protein_by_model = {}
skipped_by_model = {}
for m in models:
    per_protein_by_model[m], skipped_by_model[m] = build_per_protein(model_pred_path[m], gt_path, ATOM)

# All three models cover the same 415 test entries;
# intersect anyway so every protein is comparable across models
common_proteins = set(per_protein_by_model[models[0]].keys())
for m in models[1:]:
    common_proteins &= set(per_protein_by_model[m].keys())
common_proteins = sorted(common_proteins)

kendall_by_model = {m: [] for m in models}
skipped_short = []
for entry in common_proteins:
    gt_vals = per_protein_by_model[models[0]][entry]["gt"]
    if len(gt_vals) < MIN_VALID_RES:
        skipped_short.append(entry)
        continue
    for m in models:
        pred_vals = per_protein_by_model[m][entry]["pred"]
        tau = kendalltau(pred_vals, per_protein_by_model[m][entry]["gt"])[0]
        kendall_by_model[m].append(0.0 if np.isnan(tau) else tau)

label_map = {"lacs": "SLAE-CS-LACS", "panav": "SLAE-CS-PANAV", "noref": "SLAE-CS-BASELINE", "noref_whole": "SLAE-CS-whole BMRB"}
means, stds = {}, {}
for m in models:
    arr = np.array(kendall_by_model[m])
    means[m] = float(np.mean(arr))
    stds[m] = float(np.std(arr))
    print(f"{label_map[m]:<6} Average Kendall's tau: {means[m]:.4f}  std: {stds[m]:.4f}  n_proteins={len(arr)}")

print(f"\nSkipped (missing from a model's predictions): "
      f"{ {m: len(skipped_by_model[m]) for m in models} }")
print(f"Skipped (< {MIN_VALID_RES} valid residues): {len(skipped_short)}: {skipped_short}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })

set_style(base=19)

plot_models = ["noref", "panav", "lacs", "noref_whole"]
# x tick labels: label_map names without the "SLAE-CS" prefix
plot_labels = [label_map[m].removeprefix("SLAE-CS-").removeprefix("SLAE-CS ") for m in plot_models]
plot_means  = [means[m] for m in plot_models]
plot_stds   = [stds[m] for m in plot_models]

# distinct color per model
palette = ["#999999", "#61AFEF", "#E06C75", "#333333"]
colors = palette[:len(plot_models)]

x = np.arange(len(plot_models))

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(x, plot_means, yerr=plot_stds, capsize=4, color=colors,
              edgecolor="white", linewidth=0.6,
              error_kw=dict(elinewidth=1.2, ecolor="gray", capthick=1.2))

for bar, m in zip(bars, plot_means):
    print(f"{m:.4f}")
    # ax.text(bar.get_x() + bar.get_width() / 2, m + 0.0005,
    #         f"{m:.4f}", ha="center", va="bottom")

ax.set_xticks(x)
ax.set_xticklabels(plot_labels)
ax.set_xlabel("SLAE-CS models")

ax.set_ylim(0, 1.05)
ax.set_ylabel("Kendall's τ (mean ± std)")
# ax.set_title("Ranking correlation comparison on rBMRB-PANAV test set")
plt.tight_layout()
plt.savefig("Figs/kendalltau_comparison_seedensemble_panavtestset.png", dpi=300, bbox_inches="tight")
plt.savefig("Figs/kendalltau_comparison_seedensemble_panavtestset.svg", bbox_inches="tight")
plt.show()
